# DBT

The Data Build Tool (DBT) is a popular framework for building data manipulation systems.


## CLI

DBT is implemented in two versions:

- [`dbt-core`](https://pypi.org/project/dbt-core/): the python package that implements the CLI with which you can manipulate your project. Use `dbt` command to acces this tool.
- [DBT fusion](https://docs.getdbt.com/docs/fusion/about-fusion) new implementation of dbt that provides faster compilation of the project. Use `dbtf` command to access the fusion.

They have really similar interface which is discussed in this section.

For more check
- [Reference to the `dbt` CLI](https://docs.getdbt.com/reference/dbt-commands).
- [CLI](dbt/cli.ipynb) page.

## Structure

The DBT project suposes some kind of structure. There are corresponding directories/files:

- `/models`: the sql or python files that define what have to be done in the dbt project.
- `/seeds`: CSV-files, containing some additional information.
- `/tests`: unit-tests of the dbt project.
- `/maros`: reusable jinja templates.
- `/analysis`: files for Adhoc queries.
- `dbt_project.yml`: the main files for prject configuration.
- `profies.yml`: file that describes the credentials to connect to the database.

### Data layers

The dbt recommends organizing your transformation into three layers:

- **Staging**: the raw data received from datasources.
- **Intermediate**: calculations layer.
- **Marts**: for data marts (рус. Витрины данных).

Check more on project sctucture and naming conventions in the [How we structure out dbt prjects](https://docs.getdbt.com/best-practices/how-we-structure/1-guide-overview?version=1.12) guide from the dbt team.

## Models

In DBT, models are SQL scripts containing Jinja templates. DBT compiles these scripts into corresponding SQL code and then executes them. The results of the queries are wrapped in the corresponding DDL/DML and the results are put into the target database.

Check more in:

- [Enhance your models](https://docs.getdbt.com/docs/build/enhance-your-models?version=1.12) page of the official documentation.
- [Models](dbt/models.ipynb) dedicated page.

---

The following cell initialise the project.

In [1]:
#init
dbtf init --project-name knowledge --profile knowledge -q
cd knowledge

Info Created .vscode/extensions.json with dbt extension recommendation
Success Project created successfully!
Info Project name: knowledge
Info Project directory: knowledge
Validating profile inputs, adapters, and connection

m Debugging ⠁ [0s]                                                               


The definition of the model:

In [2]:
# file knowledge/models/script.sql
SELECT 'some data' AS col, 'new data' AS col2

This "model" simply selects hard-coded values from the database.

In [4]:
dbt run -q --select script

The table with same name then appears in the target PostgreSQL database.

**PS.** `dbt show --inline` allows to run command in the target database:

In [5]:
dbt show -q --inline "select * from public.script"

| col       | col2     |
| --------- | -------- |
| some data | new data |



## Jinja

Jinja plays a significant role in DBT, as it enables the creation of templates for SQL code that can be 'rendered' to build the specific pipelines.

The subset of facilities implemented by the dbt.

| Type                  | Examples                                   |
| --------------------- | ------------------------------------------ |
| Graph/context objects | `graph`, `model`, `this`, `target`         |
| Dependency functions  | `ref()`, `source()`                        |
| Variables/config      | `var()`, `env_var()`, `config()`           |
| Runtime state         | `execute`, `flags`, `selected_resources`   |
| Logging/debugging     | `log()`, `print()`, `debug()`              |
| SQL execution         | `run_query()`                              |
| Serialization         | `tojson`, `fromjson`, `toyaml`, `fromyaml` |
| Misc                  | `adapter`, `modules`, `builtins`           |

Check the :
- [Jinja reference](https://docs.getdbt.com/category/jinja-reference?version=1.12) page of the official documentation
- [Jinja](dbt/jinja.ipynb) page in this site.

---

Consider dbt project for example:

In [2]:
#init
dbt init knowledge --profile knowledge -q
cd knowledge

The following cell defines a dbt model that uses the `print` function within the dbt project.

In [3]:
#file knowledge/models/some_model.sql
{{ print("this message should be printed") }}
select 1;

The corresponding message is displayed when this Jinja code is executed during compilation.

In [5]:
dbt compile -q

this message should be printed


## Packages

The dbt projects can be reused by they other dbt projects. The models and macros are the main things that can be reused in dbt projects.

The packages included to the project are declared in the file `packages.yml`.

The `dbt deps` command fetches the packages and puts them in the gitignored `packages` folder.

Check the [Packages](https://docs.getdbt.com/docs/build/packages?version=2.0&name=Fusion) page of the official documentation.

---

Consider an example in which one package is used by another. The following cell creates two dbt projects:

In [1]:
#init
dbt init project1 --profile knowledge -q
dbt init project2 --profile knowledge -q

rm -r project1/models/example
rm -r project2/models/example

**Note** default examples folders have to be removed as it cause names duplication.

The model in the `package1` that would be invoked from the other one.

In [15]:
#file project1/models/the_model.sql
SELECT 'hello from project1'

The configuration that builds relation between projects:

In [16]:
#file project2/packages.yml
packages:
    - local: ../project1

Installation of the `project1` to the `project2`:

In [17]:
dbt deps --project-dir ./project2

20:32:31  Running with dbt=1.11.7
20:32:31  Installing ../project1
20:32:31  Installed from <local @ ../project1>


Model from the `package1` can be refferenced as a regular model:

In [18]:
#file project2/models/example.sql
SELECT 'the message is ' || (SELECT * FROM {{ ref('project1', 'the_model') }})

The result of the running `project2`:

In [19]:
dbt run --project-dir ./project2

20:32:38  Running with dbt=1.11.7
20:32:38  Registered adapter: postgres=1.10.0
20:32:39  Found 2 models, 464 macros
20:32:39  
20:32:39  Concurrency: 16 threads (target='dev')
20:32:39  
20:32:39  1 of 2 START sql view model public.the_model ................................... [RUN]
20:32:39  1 of 2 OK created sql view model public.the_model .............................. [CREATE VIEW in 0.11s]
20:32:39  2 of 2 START sql view model public.example ..................................... [RUN]
20:32:39  2 of 2 OK created sql view model public.example ................................ [CREATE VIEW in 0.20s]
20:32:39  
20:32:39  Finished running 2 view models in 0 hours 0 minutes and 0.47 seconds (0.47s).
20:32:39  
20:32:39  Completed successfully
20:32:39  
20:32:39  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 NO-OP=0 TOTAL=2


And the model materialized as a view:

In [21]:
dbt show --project-dir ./project2 --select example -q

| ?column?             |
| -------------------- |
| the message is he... |

